In [ ]:
using Base.Threads
println( "Number of threads: ", nthreads() )

include( "../args.jl" )
include( "../model.jl" )
include( "../geom.jl" )
include( "../recur.jl" )

loadsteps = true
savesteps = !loadsteps
loadsteps, savesteps

In [ ]:
# Case title.
δt = Δt;  ϕ = 1/4
N = 100;  ξ = 0.1;  γ = 0.1;  s = 10.0/ξ^(ϕ + 1);  τ = 75.0
case = "../data/results/case-1_crit-rho/"
pop = "N-$(N)/tau-$(round( τ, digits=6 ))/xi-$(round( ξ , digits=6 ))_spd-$(round( s, digits=6 ))/"

if ~isdir( case*pop )
    mkpath( case*pop )
end

if loadsteps
    # Number of parameter combinations.
    βlist = vcat( readdlm( case*pop*"beta-list.txt" )... )
    Nβ = length( βlist )
else
    # Draw random parameters for β and τ in the range.
    βmin = -1;  βmax = 1
    Nβ = 51;  Δβ = (βmax - βmin)/(Nβ-1)
    βlist = round.( 10.0.^(βmin:Δβ:βmax), digits=6 )  # DIMENSIONAL
end;

println( "Running simulation for $(Nβ) different values of β." )

In [ ]:
# Adapatable time-step length.
ρmin = -3;  ρmax = 1
Nρ = Nβ;  Δρ = (ρmax - ρmin)/(Nρ-1)
ρlist = round.( 10.0.^(ρmin:Δρ:ρmax), digits=6 )  # DIMENSIONAL
println( "Running for $(Nρ) different values of ρ." )

# System parameters.
μ = 1.0
L = √(N/μ)

# Activity transition variables.
η = 1/500;  γ = 0.1

# Movement variables.
λ = 0.50

# Sensing area parameter.
α = (1/4)π

# Generate parameter variable.
dparams = Params(; ρ=0.1, η=η, β=1.0, γ=γ, τ=τ, ξ=ξ, λ=λ, ϕ=ϕ, s=s, α=α )

# Create parameter sets.
dscale = Scale( 1.0, ξ )
paramsdata = [[Params(; ρ=ρ, η=η, β=β, γ=γ, τ=τ, ξ=ξ, λ=λ, ϕ=ϕ, s=s, α=α ) for ρ ∈ ρlist] for β ∈ βlist ];
nondimdata = [[Nondim( params; scale=dscale ) for params ∈ paramslist] for paramslist ∈ paramsdata];

In [ ]:
# Data folder name.
folderdata = [[case*pop*"beta-$(round( β, digits=6 ))/rho-$(round( ρ, digits=6 ))/"
    for ρ ∈ ρlist] for β ∈ βlist]

# Save default set.
if true
    saveparams( case*"default-params.json", dparams )
    savescale( case*"default-scale.json", dscale )
end;

In [ ]:
# Run simulation under each environment parameter.
T = round( defInt, 500/dscale.T );  M = 10
Tload = round( defInt, 500/dscale.T )  # If applicable.

# Compute simulation time-step.
Nt = round( defInt, T/δt );  tlist = 1:Nt
nt = round( defInt, 1/(2*δt*dscale.T) );  tsave = Set( 1:nt:Nt );

# Frequency of adjacency calculation.
δt̂ = round( defInt, 0.1/δt );

In [ ]:
# Initialize list and run optimization.
xdatadata = [[[Matrix{defFloat}( undef, length( tsave ) + 1, 3 ) for _ ∈ 1:M] for _ ∈ 1:Nρ] for _ ∈ 1:Nβ]
zdatalist = [Matrix{State}( undef, Nρ, M ) for _ ∈ 1:Nβ]
@threads for k ∈ 1:Nβ
    for i ∈ 1:Nρ
        nondim = nondimdata[k][i]
        for m ∈ 1:M
            # If steps are already saved, use as initial state.
            file = loadsteps ? folderdata[k][i]*"steps/state_T-$(Tload)_m-$(m).txt" : nothing

            # Initialize agent states.
            z = initialstate( N, L; A=1, file=file )
            ẑ = copystate( z )

            # Initialize adjacency and saved state.
            A = proximity( N, L, nondim.r, nondim.α, z.x, z.y, z.θ )
            xdatadata[k][i][m][1,:] = statecomposition( N, ẑ )

            # Run simulation.
            t̂ = 2
            for t ∈ tlist
                # Update the adjacency matrix.
                (t % δt̂) == 0 && (A = proximity( N, L, nondim.r, nondim.α, z.x, z.y, z.θ ))

                # Step simulation.
                step!( N, L, nondim, z, ẑ; A=A, δt=δt )

                # Save state if in appropriate subset.
                t ∈ tsave && (xdatadata[k][i][m][t̂,:] = statecomposition( N, ẑ ); t̂ += 1)

                # Swap contents.
                tmp = z;  z = ẑ;  ẑ = tmp
            end

            # Save last simulation state.
            zdatalist[k][i,m] = z
        end
    end
end

In [ ]:
# Compute determinism metric and related statistics.
Rdatalist = [Matrix{RecurrenceMap}( undef, Nρ, M ) for _ ∈ 1:Nβ]
ςdatalist = [Matrix{defFloat}( undef, Nρ, M ) for _ ∈ 1:Nβ]
@threads for k ∈ 1:Nβ
    for i ∈ 1:Nρ
        for m ∈ 1:M
            Rdatalist[k][i,m] = recurrence( xdatadata[k][i][m]; δx=1/100 )
            ςdatalist[k][i,m] = determinism( Rdatalist[k][i,m]; ℓ0=15 )
        end
    end
end

In [ ]:
# Determinism statistics.
ς̄data = [vcat( mean( ςdata, dims=2 )... ) for ςdata ∈ ςdatalist];

In [ ]:
# Plot the determinism in each case for visual inspection.
plt = plot( size=(400,200), xformatter=:plain, dpi=600 )

for (k, ς̄list) ∈ enumerate( ς̄data[1:10:end] )
    plot!( plt, ρlist, ς̄list; lw=2, marker=:circ, label="" )
        # label=latexstring( "τ=$(τlist[k]), β=$(βlist[k])" ) )
end

plot!( plt; xlims=(ρlist[1],ρlist[end]), xscale=:log10 )
plot!( plt; ylims=(0,1) )

plot!( plt; xlabel="spontaneous deactivation rate, "*L"ρ", ylabel="determinism", legend=:outerright )

In [ ]:
k = 1;  i = 1
println( (ς̄data[k][i], βlist[k], ρlist[i]) )

# Plot the time-series for a single parameter case and rho.
plt = plot( size=(400,200), xformatter=:plain, margin=10pt, dpi=600 )

for m ∈ 1:M
    alist = xdatadata[k][i][m][:,1]
    if m == M
        plot!( plt, δt*(0:nt:Nt), alist; color=:indianred, alpha=1, lw=2, label="" )
    else
        plot!( plt, δt*(0:nt:Nt), alist; color=:black, alpha=1/6, label="" )
    end
end

plot!( plt; xlims=(0,T), xlabel="time, "*L"t" )
plot!( plt; ylims=(0,1), ylabel="proportion of\nants active, "*L"a" )

In [ ]:
for (k, folderlist) ∈ enumerate( folderdata )
    for (i, folder) ∈ enumerate( folderlist )
        if !isdir( folder*"steps/" )
            mkpath( folder*"steps/" )
        end

        if true
            # Unpack variables of interest.
            xdata = xdatadata[k][i]
            params = paramsdata[k][i]
            writedlm( folder*"activity_T-$(round( defInt, T )).txt", [xlist[:,1] for xlist ∈ xdata] )
            writedlm( folder*"inactivity_T-$(round( defInt, T )).txt", [xlist[:,2] for xlist ∈ xdata] )
            writedlm( folder*"refractory_T-$(round( defInt, T )).txt", [xlist[:,3] for xlist ∈ xdata] )
            writedlm( folder*"determinism_T-$(round( defInt, T )).txt", ςdatalist[k][i,:] )

            # Save dimensional parameters and scale.
            saveparams( folder*"params.json", params )
            savescale( folder*"scale.json", dscale )
        end
    end
end

if true
    println( "Saving final step of simulation for future initial conditions." )

    for k ∈ 1:Nβ for i ∈ 1:Nρ for m ∈ 1:M
        folder = folderdata[k][i]
        savestate( folder*"steps/state_T-$(T)_m-$(m).txt", zdatalist[k][i,m] )
    end; end; end

    β̂list = isfile( case*pop*"beta-list.txt" ) ? vcat( readdlm( case*pop*"beta-list.txt" ), βlist ) : βlist

    # Append to existing data set if new data.
    writedlm( case*pop*"beta-list.txt", Set( β̂list ) )
end